# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata  # This is an object, not a dict
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

Let's inspect all record sets, and for each, all available fields, referencing them by their unique `@id` fields.

In [ ]:
# List all record sets and their fields
record_sets = list(dataset.record_sets())
print(f"Found {len(record_sets)} record sets.\n")
record_sets_ids = []
for rs in record_sets:
    print(f"RecordSet name: {rs.name}")
    print(f"  @id: {rs.id}")
    record_sets_ids.append(rs.id)
    if hasattr(rs, 'fields'):
        print("  Fields and columns:")
        for f in rs.fields:
            print(f"    Field name: {f.name} | @id: {f.id} | DataType: {f.data_type}")
    else:
        print("  No fields found.")
    print("")

## 3. Data Extraction
Load data from each record set into DataFrames for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Extract data from each record set
dataframes = {}

for record_set_id in record_sets_ids:
    print(f"Loading records from record set: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"  Columns: {list(df.columns)}\n  Number of rows: {len(df)}\n")
# We'll focus on the first record set by default for following examples
if len(record_sets_ids) == 0:
    raise ValueError('No record sets detected in the dataset.')
selected_record_set = record_sets_ids[0]
print(f"Selected primary record set for analysis: {selected_record_set}")
print(dataframes[selected_record_set].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes examples of removing outliers, transforming data, and grouping by key fields.

Let's select the first numeric field for demonstration.

In [ ]:
primary_df = dataframes[selected_record_set]
# Find the first numeric field by Croissant @id
numeric_field_id = None
group_field_id = None
rs_obj = None
for rs in dataset.record_sets():
    if rs.id == selected_record_set:
        rs_obj = rs
        break
if rs_obj is not None and hasattr(rs_obj, 'fields'):
    for f in rs_obj.fields:
        if f.data_type in ['Integer', 'Float', 'Number']:
            numeric_field_id = f.id
            # Pick a non-numeric field for demonstration (e.g., group by)
        if group_field_id is None and f.data_type in ['Text', 'String', 'Category']:
            group_field_id = f.id
if numeric_field_id is None:
    print("No numeric fields found in the selected record set.")
else:
    print(f"Numeric field selected: {numeric_field_id}")
    threshold = primary_df[numeric_field_id].dropna().median()
    filtered_df = primary_df[primary_df[numeric_field_id] > threshold]
    print(f"Records with {numeric_field_id} > {threshold} ({len(filtered_df)} rows):")
    print(filtered_df[[numeric_field_id]].head())
    
    norm_col = f"{numeric_field_id}_normalized"
    filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, norm_col]].head())

    if group_field_id and group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
        print(f"Grouped data by {group_field_id} (mean of numeric fields):")
        print(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot histogram of selected numeric field
if numeric_field_id:
    plt.figure(figsize=(8, 4))
    sns.histplot(primary_df[numeric_field_id].dropna(), kde=True, bins=40)
    plt.title(f"Distribution of {numeric_field_id} in record set {selected_record_set}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()
    # If group_field_id exists, show boxplot
    if group_field_id and group_field_id in primary_df.columns:
        plt.figure(figsize=(10, 6))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=primary_df)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xticks(rotation=45)
        plt.show()
else:
    print("No numeric field available for visualization.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

<!-- You can write your summary here after reviewing the output above. -->

- We loaded the Mass Spectrometry EV dataset using the Croissant schema and `mlcroissant`.
- All record sets and fields were referenced by their Croissant `@id` for robust data access.
- We explored the structure, extracted records, and performed simple statistical and visual analyses.
- For deeper analyses, consider joining with publication metadata, leveraging additional fields, or integrating with proteomic databases referenced by accession IDs.

> Use this workflow as a template for any Croissant-based dataset!